In [ ]:
import math
import h5py
# ============================================================
# 1. SETUP & CONFIGURATION
# ============================================================
import os
import numpy as np
import pandas as pd
import json
import tensorflow as tf
from pathlib import Path

# GPU Configuration
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    policy = tf.keras.mixed_precision.Policy('mixed_float16')
    tf.keras.mixed_precision.set_global_policy(policy)
    print(f'GPU mode: {len(gpus)} GPU(s), mixed_float16')
else:
    print('CPU mode')



## 2. Dataset Paths
Pointing to the downloaded CSV and JSON artifacts.

In [ ]:
# ============================================================
# 2. DATASET PATHS  (Kaggle – How2Sign-keypoints dataset)
# ============================================================
BASE_DIR = Path('/kaggle/input/datasets/nazarboholii/how2sign')

CSV_TRAIN = Path('/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_train.csv')
CSV_VAL   = Path('/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_val.csv')
CSV_TEST  = Path('/kaggle/input/datasets/nazarboholii/how2sign/how2sign_realigned_test.csv')

# JSON frames live one folder deeper than the split root
JSON_DIR_TRAIN = BASE_DIR / 'train_2D_keypoints' / 'openpose_output' / 'json'
JSON_DIR_VAL   = BASE_DIR / 'val_2D_keypoints'   / 'openpose_output' / 'json'
JSON_DIR_TEST  = BASE_DIR / 'test_2D_keypoints'  / 'openpose_output' / 'json'

for label, p in [('BASE_DIR', BASE_DIR),
                 ('CSV_TRAIN', CSV_TRAIN), ('CSV_VAL', CSV_VAL), ('CSV_TEST', CSV_TEST),
                 ('JSON_TRAIN', JSON_DIR_TRAIN), ('JSON_VAL', JSON_DIR_VAL), ('JSON_TEST', JSON_DIR_TEST)]:
    print(f"{label:12s} exists={p.exists()}  → {p}")



In [ ]:
# ============================================================
# 2.5  FEATURE EXTRACTION  (body + hands + selective face)
# ============================================================
# Final feature vector layout (232-dim):
#
#  [0   : 50 ]  body      25 kps × (x,y)          =  50
#  [50  : 92 ]  left hand 21 kps × (x,y)           =  42
#  [92  : 134]  right hand 21 kps × (x,y)           =  42
#  [134 : 232]  face       49 kps × (x,y) selective =  98
#                                               TOTAL = 232
# ============================================================

import pandas as pd
import numpy as np
import os, json, re
from pathlib import Path
from tqdm.auto import tqdm

# ── Keypoint selection ───────────────────────────────────────

FACE_KP_INDICES = (
    [17,18,19,20,21]                            +  #  5 R.eyebrow
    [22,23,24,25,26]                            +  #  5 L.eyebrow
    [36,37,38,39,40,41]                         +  #  6 R.eye
    [42,43,44,45,46,47]                         +  #  6 L.eye
    [68,69]                                     +  #  2 pupils
    [27,28,29,30]                               +  #  4 nose bridge
    [33]                                        +  #  1 nose tip
    [48,49,50,51,52,53,54,55,56,57,58,59]       +  # 12 outer lips
    [60,61,62,63,64,65,66,67]                      #  8 inner lips
)  # 49 total

NUM_BODY      = 25   # keypoints
NUM_HAND      = 21   # keypoints each hand
NUM_FACE_SEL  = 49   # selected face keypoints

# Final dims (x,y only — no confidence)
DIM_BODY      = NUM_BODY     * 2   #  50
DIM_HAND      = NUM_HAND     * 2   #  42
DIM_FACE      = NUM_FACE_SEL * 2   #  98
NUM_FEATURES  = DIM_BODY + DIM_HAND + DIM_HAND + DIM_FACE  # 232

print(f"Feature vector size: {NUM_FEATURES}")
# Body: 50 | L.Hand: 42 | R.Hand: 42 | Face: 98 | Total: 232

# ── Core extraction helpers ──────────────────────────────────

def _extract_xy(raw: list, n_kps: int) -> np.ndarray:
    """
    raw  : flat OpenPose list [x0,y0,c0, x1,y1,c1, ...]
    returns (n_kps, 2) keeping only x,y, zero-padded if short
    """
    arr = np.array(raw, dtype=np.float32).reshape(-1, 3)  # (K, 3)
    xy  = arr[:, :2]                                       # (K, 2)
    if len(xy) < n_kps:
        pad = np.zeros((n_kps - len(xy), 2), dtype=np.float32)
        xy  = np.vstack([xy, pad])
    return xy[:n_kps]                                      # (n_kps, 2)


def _extract_face_xy(raw: list) -> np.ndarray:
    """
    raw  : flat OpenPose face list (up to 70 kps × 3 = 210 values)
    returns (49, 2) — only the linguistically relevant keypoints

    Selected indices and what they capture:
      17-26  eyebrows  → question type (raised=Y/N, furrowed=WH)
      36-47  eyes      → gaze direction, eye aperture
      68-69  pupils    → gaze refinement
      27-30  nose bridge → head tilt normalization anchor
      33     nose tip  → stable center reference
      48-59  outer lips → mouth shape (mouthing words)
      60-67  inner lips → mouth openness, tongue visibility
    """
    # Parse full 70-kp array (pad if OpenPose gave fewer)
    arr = np.array(raw, dtype=np.float32).reshape(-1, 3)   # (K, 3)
    if len(arr) < 70:
        pad = np.zeros((70 - len(arr), 3), dtype=np.float32)
        arr = np.vstack([arr, pad])
    arr = arr[:70]                                          # (70, 3)

    # Select only relevant indices, keep x,y
    selected = arr[FACE_KP_INDICES, :2]                    # (49, 2)
    return selected


def parse_frame(json_path: Path) -> np.ndarray | None:
    """
    Read one OpenPose frame JSON → (232,) float32 vector.

    Vector layout:
      [0  :50 ]  body       25 × (x,y)
      [50 :92 ]  left hand  21 × (x,y)
      [92 :134]  right hand 21 × (x,y)
      [134:232]  face       49 × (x,y)  ← selective
    """
    try:
        with open(json_path) as f:
            data = json.load(f)

        people = data.get('people', [])
        p      = people[0] if people else {}

        body  = _extract_xy(
                    p.get('pose_keypoints_2d',       [0]*75), NUM_BODY)
        lhand = _extract_xy(
                    p.get('hand_left_keypoints_2d',  [0]*63), NUM_HAND)
        rhand = _extract_xy(
                    p.get('hand_right_keypoints_2d', [0]*63), NUM_HAND)
        face  = _extract_face_xy(
                    p.get('face_keypoints_2d',       [0]*210))

        return np.concatenate([
            body.flatten(),    #  50
            lhand.flatten(),   #  42
            rhand.flatten(),   #  42
            face.flatten()     #  98
        ])                     # 232 total

    except Exception:
        return None


def load_sentence(sentence_name: str, json_dir: Path) -> np.ndarray | None:
    """
    Load all frames for one sentence → (T, 232) float32.
    Handles both subfolder and flat JSON layouts.
    """
    sentence_dir = json_dir / sentence_name
    if sentence_dir.is_dir():
        frame_files = sorted(sentence_dir.glob('*.json'))
    else:
        frame_files = sorted(
            json_dir.glob(f'{sentence_name}_*_keypoints.json'))

    if not frame_files:
        return None

    frames = [v for fp in frame_files
              if (v := parse_frame(fp)) is not None]

    return np.stack(frames).astype(np.float32) if frames else None


def get_available_sentences(json_dir: Path) -> set:
    """Discover sentence names from subfolder names or filenames."""
    subdirs = [d.name for d in json_dir.iterdir() if d.is_dir()]
    if subdirs:
        print(f'    📁 {len(subdirs):,} sentence sub-folders')
        return set(subdirs)
    pattern   = re.compile(r'^(.+)_\d{12}_keypoints\.json$')
    sentences = {m.group(1) for fp in json_dir.glob('*.json')
                 if (m := pattern.match(fp.name))}
    print(f'    📄 {len(sentences):,} unique sentences (flat layout)')
    return sentences


def build_subset_and_extract(csv_path, json_dir: Path,
                              out_dir, out_csv,
                              subset_size, split_label):
    print(f"\n{'='*60}")
    print(f'  {split_label}  ←  {csv_path}')
    print(f"{'='*60}")

    if not os.path.exists(csv_path):
        print('  ❌  CSV not found'); return None

    # ── Load CSV ─────────────────────────────────────────────
    df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
    if 'SENTENCE_NAME' not in df.columns:
        df = pd.read_csv(csv_path, on_bad_lines='skip')
    print(f'  📄  CSV rows: {len(df):,}')

    # ── Discover available JSON sentences ────────────────────
    if not json_dir.exists():
        print(f'  ❌  JSON dir not found: {json_dir}'); return None
    available = get_available_sentences(json_dir)
    if not available:
        print('  ❌  No sentences found'); return None

    # ── Match & sample ───────────────────────────────────────
    df_matched = df[df['SENTENCE_NAME'].astype(str).isin(available)].copy()
    print(f'  ✅  Matched: {len(df_matched):,} / {len(df):,}')
    if df_matched.empty:
        print('  ❌  0 matches'); return None

    df_sampled = (df_matched.sample(n=subset_size, random_state=42)
                  if len(df_matched) > subset_size
                  else df_matched).reset_index(drop=True)
    print(f'  🎲  Subset: {len(df_sampled):,}')

    # ── Extract JSON → .npy ──────────────────────────────────
    os.makedirs(out_dir, exist_ok=True)
    npy_paths, skipped = [], 0

    for _, row in tqdm(df_sampled.iterrows(),
                       total=len(df_sampled),
                       desc=f'  {split_label}'):
        sname   = str(row['SENTENCE_NAME'])
        npy_out = os.path.join(out_dir, f'{sname}.npy')

        if not os.path.exists(npy_out):
            seq = load_sentence(sname, json_dir)
            if seq is not None:
                np.save(npy_out, seq)            # (T, 232)
            else:
                skipped += 1
                npy_paths.append(None)
                continue

        npy_paths.append(npy_out)

    # ── Save enriched CSV ────────────────────────────────────
    df_sampled['NPY_PATH'] = npy_paths
    df_final = df_sampled[df_sampled['NPY_PATH'].notna()].reset_index(drop=True)
    df_final.to_csv(out_csv, index=False)

    print(f'  💾  {len(df_final):,} .npy files  →  {out_dir}/')
    print(f'  ⏭️   Skipped: {skipped}')
    print(f'  📝  CSV     →  {out_csv}')
    return out_csv


# ── Run ──────────────────────────────────────────────────────
CSV_TRAIN = build_subset_and_extract(
    str(BASE_DIR / 'how2sign_realigned_train.csv'), JSON_DIR_TRAIN,
    'train_features', 'how2sign_train_subset.csv', 6000, 'TRAIN')

CSV_VAL = build_subset_and_extract(
    str(BASE_DIR / 'how2sign_realigned_val.csv'), JSON_DIR_VAL,
    'val_features', 'how2sign_val_subset.csv', 2000, 'VAL')

CSV_TEST = build_subset_and_extract(
    str(BASE_DIR / 'how2sign_realigned_test.csv'), JSON_DIR_TEST,
    'test_features', 'how2sign_test_subset.csv', 2000, 'TEST')